# Quickstart: run a single simulation

Run a baseline simulation, inspect outputs, and plot a snapshot.


## Setup


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import matplotlib.pyplot as plt

repo_root = Path().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

# Helper to run CLI modules with correct PYTHONPATH
_env = os.environ.copy()
_env["PYTHONPATH"] = str(repo_root / "src")

def run_module(module, args):
    cmd = [sys.executable, "-m", module] + list(args)
    return subprocess.run(cmd, check=True, env=_env)

print("repo_root:", repo_root)


## Run a baseline config
This calls the core simulation function directly (no CLI).


In [ ]:
from skin_diffusion.config import load_config
from skin_diffusion.run_utils import run_simulation

cfg = load_config("configs/sim/v1_baseline.yaml")
C_snap, t_save, D_field, k_field, patch_mask, diagnostics, metrics, stability_info = run_simulation(cfg)

print("C_snap shape:", C_snap.shape)
print("t_save shape:", t_save.shape)
print("D shape:", D_field.shape)
print("patch_mask true count:", int(patch_mask.sum()))
print("metrics keys:", sorted(metrics.keys()))


## Plot snapshots
We show the first, middle, and last saved frames.


In [ ]:
import matplotlib.pyplot as plt

idxs = [0, len(t_save) // 2, len(t_save) - 1]
for idx in idxs:
    plt.figure()
    plt.imshow(C_snap[idx], origin="upper", aspect="auto")
    plt.colorbar()
    plt.title(f"t={t_save[idx]:.4f}")
    plt.show()


## Plot depth profiles
These profiles show x-averaged concentration vs depth.


In [ ]:
import numpy as np

for idx in idxs:
    depth = np.arange(C_snap.shape[1])
    profile = C_snap[idx].mean(axis=1)
    plt.figure()
    plt.plot(profile, depth)
    plt.gca().invert_yaxis()
    plt.xlabel("C (x-avg)")
    plt.ylabel("depth index")
    plt.title(f"Depth profile at t={t_save[idx]:.4f}")
    plt.show()
